# သင်ခန်းစာ 11 - Model Context Protocol (MCP)

**Model Context Protocol (MCP)** သည် အေးဂျင့်များကို runtime တွင် အလိုအလျောက် ကိရိယာများ၊ အရင်းအမြစ်များနှင့် ဒေတာရင်းမြစ်များကို ရှာဖွေ၍ အသုံးပြုနိုင်စေသည့် ဖွင့်လှစ် စံတစ်ခု ဖြစ်သည်။ ကိရိယာများကို အေးဂျင့်အတွင်း တင်းကြပ်စွာ ကုဒ်ထည့်သွင်းခြင်းမှ ရှောင်ကင်းနိုင်ပြီး၊ MCP က အလိုအလျောက် စွမ်းရည်များကို ထုတ်ဖော်ပြသသည့် ပြင်ပ ဆာဗာများနှင့် အေးဂျင့်များကို ချိတ်ဆက်ခွင့် ပြုစေသည်။

ဒီသင်ခန်းစာတွင် သင်လေ့လာမည့်အချက်များ:
- MCP ဆိုတာဘာလဲ နှင့် အေးဂျင့်စနစ်များအတွက် ဘာကြောင့် အရေးပါသလဲ
- MCP ၏ client-server ဖွဲ့စည်းပုံ မည်သို့ အလုပ်လုပ်သည်
- MCP ပုံစံ tool discovery ကို အသုံးပြုသော အေးဂျင့်များကို မည်သို့ တည်ဆောက်ရမည်


## ဆက်တင်

**လိုအပ်ချက်များ:**
- တပ်ဆင်ထားသော မော်ဒယ်တစ်ခုပါရှိသည့် Azure AI Foundry စီမံကိန်း
- အတည်ပြုရန် `az login` ကို အသုံးပြုပါ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) ဆိုတာဘာလဲ?

MCP သည် AI အေဂျင့်များအတွက် ပြင်ပ ကိရိယာများနှင့် ဒေတာရင်းမြစ်များကို ရှာဖွေ၍ အပြန်အလှန် ဆက်သွယ်နိုင်ရန် စံနည်းလမ်းတစ်ခုကို သတ်မှတ်ပေးသည်။

- **MCP Server**: စံပြပရိုတိုကောတစ်ခုမှတဆင့် ကိရိယာများ၊ အရင်းအမြစ်များနှင့် prompt များကို ဖော်ထုတ်ပေးသည်။
- **MCP Client**: ဆာဗာများနှင့် ချိတ်ဆက်ကာ ရနိုင်သော စွမ်းရည်များကို ရှာဖွေတွေ့ရှိပေးသော အေဂျင့် runtime
- **Dynamic Discovery**: အေဂျင့်များသည် ကိရိယာများကို အရင်ချိတ်ဆက်သတ်မှတ်ထားစရာ မလိုဘဲ — runtime အတွင်း ရနိုင်သမျှကို ရှာဖွေတွေ့ရှိနိုင်သည်။

ဤသည်က အေဂျင့်ကုဒ်ကို ပြင်ဆင်စရာမလိုဘဲ အသစ်သော စွမ်းရည်များ ထည့်သွင်းနိုင်သော ကျယ်ပြန့်ချဲ့ထွင်နိုင်သည့် အေဂျင့်စနစ်များ တည်ဆောက်ရာတွင် အလွန်အစွမ်းထက်သည်။


## MCP အလုပ်လုပ်ပုံ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. အေးဂျင့် (MCP client) သည် MCP server သို့ ချိတ်ဆက်သည်
2. ဆာဗာသည် ရနိုင်သော ကိရိယာများနှင့် ၎င်းတို့၏ schemas များကို စာရင်းအဖြစ် ပြန်လှန်ပေးသည်
3. အေးဂျင့်သည် မိမိ၏ စဉ်းစားမှုအတွင်း တွေ့ရှိထားသော မည်သည့် ကိရိယာကိုမဆို ခေါ်ယူနိုင်သည်
4. ရလဒ်များကို တူညီသော ပရိုတိုကောဖြင့် ပြန်လည် ပို့ဆောင်သည်


## MCP ကိရိယာ ရှာဖွေရေးကို အတုအယောင် ပြသခြင်း

တကယ့် MCP ဆာဗာသည် လည်ပတ်နေသော ဆာဗာလုပ်ငန်းစဉ်တစ်ခု လိုအပ်သဖြင့်၊ ကျွန်တော်တို့သည် MCP-နှင့် ချိတ်ဆက်ထားသော ဧည့်ခံဝန်ဆောင်မှုမှ ပေးဆောင်နိုင်မည့်အရာများကို အတုဖော်ဆောင်သလို `@tool` လုပ်ဆောင်ချက်များကို အသုံးပြုပြီး ဤနမူနာကို ပြသပါမည်။

ထုတ်လုပ်ရေးတွင်၊ ဤကိရိယာများကို ဒေသတွင်းတွင် သတ်မှတ်ထားခြင်းမဟုတ်ဘဲ MCP ဆာဗာမှ အလိုအလျော့ရရှိစွာ ရှာဖွေတွေ့ရှိမည် ဖြစ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-စတိုင် ကိရိယာများနှင့် အေးဂျင့် တည်ဆောက်ခြင်း


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ထုတ်လုပ်မှုတွင် MCP

ထုတ်လုပ်မှုပတ်ဝန်းကျင်တွင်၊ MCP သည် စွမ်းဆောင်ရည်မြင့် နမူနာများကို ခွင့်ပြုသည်:

- **ပြောင်းလဲနိုင်သော ကိရိယာ ရှာဖွေခြင်း**: Agent များသည် MCP ဆာဗာများနှင့် ချိတ်ဆက်ပြီး runtime အချိန်တွင် ကိရိယာများကို ရှာဖွေတွေ့ရှိနိုင်သည်။
- **ခွဲထားသော ဖွဲ့စည်းပုံ**: ကိရိယာ ပံ့ပိုးသူများသည် Agent ထံမှ လွတ်လပ်စွာ အပ်ဒိတ်များ ပြုလုပ်နိုင်သည်။
- **အဖွဲ့အစည်းများ ကြား မျှဝေခြင်း**: အဖွဲ့များသည် MCP ဆာဗာများမှတဆင့် မည်သူ့ Agent မဆို အသုံးပြုနိုင်သော အရည်အချင်းများကို ဖော်ပြနိုင်သည်။
- **Microsoft Agent Framework ထောက်ပံ့မှု**: MAF တွင် `mcp` အဆက်သွယ်မှုမှတဆင့် ထည့်သွင်းထားသော MCP client ထောက်ပံ့မှု ရှိသည်။

MAF နှင့် အမှန်တကယ် MCP ဆာဗာကို အသုံးပြုရန် `hosted_mcp_tool()` သို့မဟုတ် MCP client အဆက်သွယ်မှုမှတဆင့် ချိတ်ဆက်ရမည်။

**နောက်ထပ်သိရှိရန်:**
- [MCP သတ်မှတ်ချက်](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ထောက်ပံ့မှု](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## အကျဉ်းချုပ်

ဒီသင်ခန်းစာတွင် သင်သည် အောက်ပါအချက်များကို သင်ယူခဲ့သည်:
- **MCP** သည် agent များနှင့် ကိရိယာ ပံ့ပိုးသူများအကြား ကိရိယာများကို runtime တွင် ဖော်ထုတ်နိုင်ရန် ဖွင့်လှစ်ထားသည့် စံသတ်မှတ်ချက်တစ်ခု ဖြစ်သည်
- ဒီ **client-server architecture** သည် agent များအား runtime အချိန်တွင် စွမ်းဆောင်ရည်များကို ရှာဖွေခွင့် ပေးသည်
- MCP သည် **ချဲ့ထွင်နိုင်ပြီး ခွဲထုတ်ထားသော agent စနစ်များ** ကို အားပေးကူညီပြီး ကုဒ်ပြောင်းလဲမှု မရှိဘဲ ကိရိယာများ ထပ်ထည့်နိုင်စေသည်
- Microsoft Agent Framework သည် ထုတ်လုပ်မှုပုံစံ အသုံးပြုခြင်းအတွက် **ပေါင်းစည်းထည့်သွင်းထားသော MCP ပံ့ပိုးမှု** ကို ပံ့ပိုးပေးသည်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
သတိပေးချက်:
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ကို အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့ မှန်ကန်မှုအတွက် အကောင်းဆုံး ကြိုးပမ်းပါသော်လည်း အလိုအလျောက် ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှန်ကန်မှုမပြည့်စုံမှုများ ရှိနိုင်ကြောင်း သိရှိထားရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူလဘာသာစကားဖြင့်သာ တရားဝင်အရင်းမြစ် အဖြစ် ယူဆသင့်ပါသည်။ အရေးကြီးသော အချက်အလက်များအတွက် လူသား ပရော်ဖက်ရှင်နယ် ဘာသာပြန်လုပ်ငန်းကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုသဖြင့် ဖြစ်ပေါ်နိုင်သည့် နားမလည်မှုများ သို့မဟုတ် မမှန်ကန်သော အဓိပ္ပာယ်ဖွင့်ချက်များအတွက် ကျွန်ုပ်တို့ တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
